In [ ]:
# Run this cell if using Google Colab or if pyapprox is not installed locally
try:
    import pyapprox
except ImportError:
    !pip install "pyapprox[all]" -q
    # If the PyPI install fails, uncomment the line below to install from source:
    # !pip install "pyapprox[all] @ git+https://github.com/sandialabs/pyapprox.git" -q



## Learning Objectives

After completing this tutorial, you will be able to:

- Explain why a correlated low-fidelity model can reduce the variance of a Monte Carlo (MC) estimator
- Write down the Control Variate Monte Carlo (CVMC) estimator and identify its free parameter $\eta$
- State the optimal $\eta$ and the resulting variance reduction in terms of model correlation
- Identify when CVMC helps and when it does not

## Prerequisites

Complete [Monte Carlo Sampling](monte_carlo_sampling.ipynb) and
[Estimator Accuracy and MSE](estimator_accuracy_mse.ipynb) before this tutorial.

## Motivation

The [Estimator Accuracy](estimator_accuracy_mse.ipynb) tutorial showed that the variance of the
MC mean estimator is $\sigma^2_\alpha / N$, where $\sigma^2_\alpha$ is the variance of the
high-fidelity model output $f_\alpha$ and $N$ is the number of samples. Reducing this variance
requires either more samples --- which is expensive --- or a smarter estimator.

Control Variate Monte Carlo (CVMC) is the simplest example of a smarter estimator. The idea: if
we have access to a cheap model $f_\kappa$ that is **correlated** with $f_\alpha$ and whose mean
$\mu_\kappa = \mathbb{E}_\theta[f_\kappa(\boldsymbol{\theta})]$ is **known exactly**, we can use
the cheap model to cancel a large fraction of the MC error.

@fig-model-surfaces illustrates this with two models of a 2D input. The surface plots show that
$f_\alpha$ and $f_\kappa$ have similar shape --- when one is large, so is the other. The scatter
plot on the right confirms that model outputs are tightly correlated ($\rho \approx 0.9$).

In [ ]:
#| echo: false
#| fig-cap: "Left: overlaid response surfaces of $f_\\alpha$ (blue) and $f_\\kappa$ (orange) over $[-1,1]^2$, showing similar shape with a gap between them. Right: scatter plot of 100 random evaluations confirming tight output correlation $\\rho$."
#| label: fig-model-surfaces

import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox.benchmarks.instances.multifidelity.tunable_ensemble import (
    tunable_ensemble_3model,
)
from pyapprox.tutorial_figures._cv_acv import plot_model_surfaces

bkd = NumpyBkd()
np.random.seed(0)
benchmark = tunable_ensemble_3model(bkd, theta1=np.pi / 2 * 0.95)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122)
plot_model_surfaces(benchmark, bkd, ax1, ax2)
plt.tight_layout()
plt.show()

## The CVMC Estimator

The standard MC estimator of $\mu_\alpha = \mathbb{E}_\theta[f_\alpha(\boldsymbol{\theta})]$ is

$$
\hat{\mu}_\alpha = \frac{1}{N} \sum_{k=1}^{N} f_\alpha(\boldsymbol{\theta}^{(k)}).
$$

The CVMC estimator adds a correction term built from $f_\kappa$:

$$
\hat{\mu}_\alpha^{\text{CV}} = \hat{\mu}_\alpha + \eta \left( \hat{\mu}_\kappa - \mu_\kappa \right)
$$ {#eq-cvmc}

where $\hat{\mu}_\kappa = \frac{1}{N}\sum_{k=1}^N f_\kappa(\boldsymbol{\theta}^{(k)})$ is the
MC mean of the low-fidelity model evaluated on the **same** $N$ samples, and $\eta$ is a scalar
weight we are free to choose.

The correction term $\hat{\mu}_\kappa - \mu_\kappa$ has **mean zero** --- it is pure MC error
in the low-fidelity estimate. Adding it to $\hat{\mu}_\alpha$ does not introduce bias.
But if the errors in $\hat{\mu}_\alpha$ and $\hat{\mu}_\kappa$ are correlated, choosing $\eta$
with the right sign causes the correction to partially cancel the error in $\hat{\mu}_\alpha$.


## Variance Reduction Depends on Correlation

The key result (derived in [Control Variate Analysis](control_variate_analysis.ipynb)) is that
with the optimal choice of $\eta$, the CVMC estimator variance satisfies

$$
\mathbb{V}[\hat{\mu}_\alpha^{\text{CV}}] = \mathbb{V}[\hat{\mu}_\alpha] \left(1 - \rho^2_{\alpha\kappa}\right)
$$ {#eq-variance-reduction}

where $\rho_{\alpha\kappa} = \mathrm{Corr}(f_\alpha, f_\kappa)$ is the correlation between the
two model outputs. The factor $(1 - \rho^2_{\alpha\kappa})$ is always between 0 and 1, so CVMC
always reduces (or at worst equals) the MC variance.

@fig-variance-reduction-vs-rho shows this relationship. Near-perfectly correlated models
($|\rho| \approx 1$) can reduce variance by orders of magnitude; uncorrelated models
($\rho \approx 0$) offer no benefit.

In [ ]:
#| echo: false
#| fig-cap: "CVMC variance reduction factor $(1 - \\rho^2)$ as a function of model correlation $\\rho$. A correlation of $|\\rho| = 0.9$ reduces variance by $81\\%$; $|\\rho| = 0.99$ reduces it by $99\\%$."
#| label: fig-variance-reduction-vs-rho

import matplotlib.pyplot as plt
from pyapprox.tutorial_figures._cv_acv import plot_variance_reduction_vs_rho

fig, ax = plt.subplots(figsize=(7, 4))
plot_variance_reduction_vs_rho(ax)
plt.tight_layout()
plt.show()

## What CVMC Looks Like in Practice

@fig-cvmc-histograms shows the distribution of 1000 independent MC and CVMC mean estimates
for a pair of models with $\rho \approx 0.9$. Both estimators are centered on the true mean,
confirming that CVMC is unbiased. But the CVMC histogram is dramatically narrower.

In [ ]:
#| echo: false
#| fig-cap: "Distribution of 1000 independent MC and CVMC estimates of $\\mu_\\alpha$. Both are centered on the true mean (black line), but CVMC has far smaller spread. The low-fidelity model has correlation $\\rho \\approx 0.9$ with the high-fidelity model."
#| label: fig-cvmc-histograms

import numpy as np
import matplotlib.pyplot as plt
from pyapprox.util.backends.numpy import NumpyBkd
from pyapprox.benchmarks.instances.multifidelity.tunable_ensemble import (
    tunable_ensemble_3model,
)
from pyapprox.tutorial_figures._cv_acv import plot_cvmc_histograms

bkd = NumpyBkd()
np.random.seed(0)
benchmark = tunable_ensemble_3model(bkd, theta1=np.pi / 2 * 0.95)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
N, rho, n_trials = plot_cvmc_histograms(benchmark, bkd, axes)
fig.suptitle(
    rf"$N = {N}$, $\rho = {rho:.2f}$, {n_trials} independent trials",
    fontsize=11,
)
plt.tight_layout()
plt.show()

## When Does CVMC Help?

CVMC requires two ingredients:

1. **A low-fidelity model $f_\kappa$ with a known mean $\mu_\kappa$.** If $\mu_\kappa$ is not
   known analytically, it must be estimated --- introducing additional error. That case is handled
   by [Approximate Control Variates](acv_concept.ipynb).

2. **High correlation $|\rho_{\alpha\kappa}|$ between the models.** The variance reduction
   $(1 - \rho^2)$ is only substantial when $|\rho| \gtrsim 0.5$. A weakly correlated low-fidelity
   model offers little benefit.

The cost of a single CVMC estimate is identical to plain MC: $N$ evaluations of $f_\alpha$ and
$N$ evaluations of $f_\kappa$ on the same samples. The reduction in estimator variance is entirely
due to cancellation of correlated errors, not a reduction in work.


## Key Takeaways

- CVMC adds a zero-mean correction $\eta(\hat{\mu}_\kappa - \mu_\kappa)$ to the MC estimator;
  the correction is unbiased by construction
- With the optimal $\eta$, the variance reduction factor is $(1 - \rho^2_{\alpha\kappa})$,
  determined entirely by the correlation between models
- $|\rho| \approx 1$ gives near-perfect variance cancellation; $\rho \approx 0$ gives no benefit
- CVMC requires $\mu_\kappa$ to be known; when it is not, use
  [Approximate Control Variates](acv_concept.ipynb)


::: {.callout-tip}
Ready to try this? See [API Cookbook → CVEstimator](multifidelity_estimation_cookbook.qmd#estimator-quick-reference).
:::

## Exercises

1. If $\rho = 0.7$, by what factor does CVMC reduce the variance compared to plain MC?
   How many fewer samples would you need to achieve the same standard error?

2. Suppose $f_\kappa$ is the true model $f_\alpha$ itself (i.e., $\rho = 1$). What does
   $\hat{\mu}^{\text{CV}}_\alpha$ reduce to? Is this useful in practice?

3. The correction term is $\eta(\hat{\mu}_\kappa - \mu_\kappa)$. Explain in words why a negative
   $\eta$ is appropriate when the models are positively correlated ($\rho > 0$).


## Next Steps

- [Control Variate Analysis](control_variate_analysis.ipynb) --- Derive the optimal $\eta$ and
  the $1 - \rho^2$ result from first principles
- [API Cookbook](multifidelity_estimation_cookbook.qmd#estimator-quick-reference) --- Use the PyApprox CVMC API on a
  real model
- [Approximate Control Variates](acv_concept.ipynb) --- What to do when $\mu_\kappa$ is unknown